# 🎬 زیرنویس فارسی و برش ریلز روی Google Colab

این نوت‌بوک همان گردش‌کار دومرحله‌ای ریپو را با GPU کولب انجام می‌دهد:

1. استخراج زیرنویس از صدای واضح ویدیو با Faster-Whisper؛
2. پیشنهاد نقاط برش بر اساس مکث دیالوگ، سکوت صوتی و پایان جمله؛
3. ترجمهٔ فارسی — دستی/با دستیار، یا خودکار با API اگر کلید را در Colab Secrets گذاشته باشید؛
4. بازبینی برش‌ها توسط شما؛
5. چسباندن فارسی بالای انگلیسی سوخته و ساخت نسخهٔ کامل + قسمت‌ها.

> از منوی Runtime گزینهٔ Change runtime type را روی GPU بگذارید. فایل ویدیو را در خود ریپوی گیت‌هاب آپلود نکنید؛ اینجا مستقیم در کولب آپلود می‌شود.


## ۱) نصب ریپو و FFmpeg

این سلول ریپوی واقعی پروژه را می‌گیرد. اگر دوباره اجرایش کنید، آخرین نسخهٔ شاخهٔ فعلی را هم دریافت می‌کند.


In [ ]:
!test -d subtitle-agent || git clone https://github.com/maryamghabel3-debug/subtitle-agent.git
%cd subtitle-agent
!git pull --ff-only
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -e ".[whisper]"
!nvidia-smi || true
!fa-subs doctor


## ۲) تنظیمات و وارد کردن ویدیو

دو راه برای رساندن ویدیو به کولب داری:

- **آپلود مستقیم (`upload`)**: ساده‌ترین راه برای تست و فایل‌های کوچک‌تر. وقتی سلول بعد اجرا شود، پنجره انتخاب فایل باز می‌شود و ویدیویی را که از تلگرام روی سیستمت داری انتخاب می‌کنی. این فایل موقت است و با بسته‌شدن نشست کولب پاک می‌شود.
- **گوگل‌درایو (`drive`)**: برای ویدیوهای حجیم و کارهای تکراری بهتر است. اول در گوگل‌درایو پوشه‌ای مثل `SubtitleVideos` بساز و ویدیو را آنجا آپلود کن. بعد در سلول تنظیمات `INPUT_METHOD = "drive"` و مسیر کامل فایل را در `DRIVE_VIDEO_PATH` بنویس. کولب از تو اجازه دسترسی به درایو می‌خواهد.

تنظیمات مهم:

- `SOURCE_LANGUAGE` زبان واقعی گفتار است؛ مثلاً `en` برای انگلیسی، `zh` برای چینی و `auto` برای تشخیص خودکار.
- برای GPU کولب، `large-v3` انتخاب دقیق‌تری است؛ اگر حافظه یا زمان کم بود `medium` یا `small` را انتخاب کن.
- اگر از قبل زیرنویس زبان مبدأ داری، مسیر یا نامش را در `SOURCE_SRT` بگذار؛ وگرنه `None` بماند.
- کلید API را داخل کد ننویس. برای ترجمه خودکار، در پنل Secrets کولب یک راز با نام `OPENAI_API_KEY` بساز و دسترسی نوت‌بوک به آن را فعال کن.


In [ ]:
import os
from pathlib import Path

INPUT_METHOD = "upload"              # upload = انتخاب فایل از سیستم | drive = خواندن از گوگل‌درایو
VIDEO = "video.mp4"                  # در حالت upload؛ اگر تنها فایل ویدیویی باشد خودکار پیدا می‌شود
DRIVE_VIDEO_PATH = "/content/drive/MyDrive/SubtitleVideos/video.mp4"  # در حالت drive مسیر کامل فایل
DRIVE_SOURCE_SRT = None              # مسیر SRT مبدأ در درایو، اگر داری؛ وگرنه None
SOURCE_SRT = None                    # مثال: "video.source.srt"؛ اگر ندارید None
SOURCE_LANGUAGE = "en"               # en | zh | ja | ko | auto | ...
WHISPER_MODEL = "large-v3"           # large-v3 | medium | small | base

PART_TARGET = 90                     # طول هدف هر قسمت، ثانیه
PART_MIN = 45
PART_MAX = 150

TRANSLATION_MODE = "manual"          # manual = ترجمه با شما/دستیار | auto = API اگر کلید بود
TRANSLATION_MODEL = "gpt-4o-mini"
TRANSLATION_API_BASE = "https://api.openai.com/v1"

MAKE_FULL = True
POSITION = "above_english"           # فارسی بالای انگلیسی سوخته
MARGIN_V = None                      # اگر لازم شد عدد پیکسل بدهید
PRESET = "veryfast"
CRF = 21

try:
    from google.colab import userdata
    _api_key = userdata.get("OPENAI_API_KEY")
    if _api_key:
        os.environ["OPENAI_API_KEY"] = _api_key
        print("✅ کلید API از Colab Secrets خوانده شد (مقدارش نمایش داده نمی‌شود).")
    else:
        print("ℹ️ کلید API در Secrets نیست؛ حالت دستی در دسترس است.")
except Exception:
    print("ℹ️ Colab Secrets در دسترس نیست؛ حالت دستی در دسترس است.")

from google.colab import files

uploaded = {}
if INPUT_METHOD == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = DRIVE_VIDEO_PATH
    if DRIVE_SOURCE_SRT:
        SOURCE_SRT = DRIVE_SOURCE_SRT
    if not Path(VIDEO).exists():
        raise FileNotFoundError(
            f"ویدیو در گوگل‌درایو پیدا نشد: {VIDEO}. "
            "نام پوشه و نام دقیق فایل را در DRIVE_VIDEO_PATH بررسی کن؛ "
            "مسیر معمول از /content/drive/MyDrive/ شروع می‌شود."
        )
    print("🎬 ویدیو از گوگل‌درایو آماده است:", VIDEO)
elif INPUT_METHOD == "upload":
    uploaded = files.upload()
    print("فایل‌های آپلودشده:", list(uploaded))
    if not Path(VIDEO).exists():
        video_candidates = [
            name for name in uploaded
            if Path(name).suffix.lower() in {".mp4", ".mkv", ".mov", ".webm", ".avi"}
        ]
        if len(video_candidates) == 1:
            VIDEO = video_candidates[0]
            print("🎬 ویدیو به‌صورت خودکار انتخاب شد:", VIDEO)
else:
    raise ValueError('INPUT_METHOD باید "upload" یا "drive" باشد.')


## ۳) مرحلهٔ آماده‌سازی: زیرنویس مبدأ + برش‌های پیشنهادی

این سلول سه فایل می‌سازد و آن‌ها را در یک ZIP برای دانلود آماده می‌کند:

- `*.source.srt`: متن استخراج‌شده از گفتار؛
- `*.fa.srt`: قالب فارسی با همان زمان‌بندی؛ اگر حالت `auto` و کلید API فعال باشد، ترجمه‌شده است؛
- `*.cuts.suggested.txt`: پایان پیشنهادی هر قسمت همراه با دلیل و آخرین دیالوگ.


In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path

video_path = Path(VIDEO)
if not video_path.exists():
    raise FileNotFoundError(f"ویدیو پیدا نشد: {VIDEO}؛ نام فایل آپلودشده را در VIDEO اصلاح کنید.")

stem = video_path.stem
job_dir = Path("job")
job_dir.mkdir(exist_ok=True)
source_srt_path = job_dir / f"{stem}.source.srt"
fa_srt_path = job_dir / f"{stem}.fa.srt"
cuts_path = job_dir / f"{stem}.cuts.suggested.txt"

if SOURCE_SRT:
    shutil.copy(SOURCE_SRT, source_srt_path)
    print("✅ از زیرنویس مبدأ آماده استفاده شد:", SOURCE_SRT)
else:
    subprocess.run([
        "fa-subs", "transcribe", str(video_path),
        "--out", str(source_srt_path),
        "--language", SOURCE_LANGUAGE,
        "--whisper-model", WHISPER_MODEL,
    ], check=True)

subprocess.run([
    "fa-subs", "suggest-cuts", str(video_path),
    "--srt", str(source_srt_path),
    "--out", str(cuts_path),
    "--target", str(PART_TARGET),
    "--min-duration", str(PART_MIN),
    "--max-duration", str(PART_MAX),
], check=True)

AUTO_TRANSLATED = False
if TRANSLATION_MODE == "auto" and os.environ.get("OPENAI_API_KEY"):
    import yaml
    translation_config_path = job_dir / "translation-config.yaml"
    translation_config_path.write_text(yaml.safe_dump({
        "source_language": SOURCE_LANGUAGE,
        "target_language": "fa",
        "translation_provider": "auto",
        "translation_model": TRANSLATION_MODEL,
        "translation_api_base": TRANSLATION_API_BASE,
        "translation_api_key_env": "OPENAI_API_KEY",
        "translation_batch_size": 20,
    }, allow_unicode=True, sort_keys=False), encoding="utf-8")
    subprocess.run([
        "fa-subs", "translate", str(source_srt_path),
        "--out", str(fa_srt_path),
        "--provider", "auto",
        "--config", str(translation_config_path),
    ], check=True)
    AUTO_TRANSLATED = True
    print("✅ ترجمه با API انجام شد؛ حتماً نمونه‌خوانی کنید.")
else:
    shutil.copy(source_srt_path, fa_srt_path)
    print("✋ قالب فارسی ساخته شد؛ متن‌ها هنوز زبان مبدأ هستند و باید ترجمه شوند.")

prep_zip = Path("prep-files.zip")
with zipfile.ZipFile(prep_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in (source_srt_path, fa_srt_path, cuts_path):
        zf.write(path, arcname=path.name)
files.download(str(prep_zip))
print("📦 فایل‌های آماده‌سازی دانلود شدند:", prep_zip)


## ۴) بازبینی شما: ترجمه و تأیید برش‌ها

### زیرنویس فارسی

اگر ترجمه خودکار انجام نشده است، در فایل `*.fa.srt` فقط متن هر خط را فارسی کنید. شماره‌ها و زمان‌بندی‌ها نباید تغییر کنند. می‌توانید فایل مبدأ یا قالب را برای دستیار در چت بفرستید تا مستقیم به فارسی برگرداند.

### برش‌ها

فایل `*.cuts.suggested.txt` را باز کنید. هر خط پایان یک قسمت است. می‌توانید زمان را جابه‌جا کنید، خطی را پاک کنید تا دو قسمت یکی شوند، یا خط تازه‌ای با قالب `HH:MM:SS.mmm` اضافه کنید. توضیحات بعد از `#` خوانده نمی‌شوند.

> اگر همین نشست کولب هنوز باز است، می‌توانید فایل داخل پوشهٔ `job` را از پنل Files ویرایش کنید. اگر نشست بسته شده، سلول‌های نصب و تنظیمات را دوباره اجرا و فایل‌های نهایی را آپلود کنید؛ مرحلهٔ آماده‌سازی را لازم نیست تکرار کنید، به شرطی که ویدیو و فایل‌های نهایی را دوباره در دسترس بگذارید.


## ۵) آپلود فایل فارسی نهایی و برش تأییدشده

فایل فارسی نهایی و در صورت نیاز فایل برش ویرایش‌شده را آپلود کنید و نام دقیقشان را در دو متغیر زیر بگذارید.

- اگر ترجمهٔ API در مرحلهٔ ۳ انجام شده و همان را قبول دارید، `FA_SRT_FINAL = None` بماند.
- اگر برش داخل پوشهٔ `job` را در همین نشست ویرایش کرده‌اید، `APPROVED_CUTS = None` بماند.


In [ ]:
FA_SRT_FINAL = None       # مثال: "نام_ویدیو.fa.srt"
APPROVED_CUTS = None      # مثال: "نام_ویدیو.cuts.suggested.txt"

uploaded_final = files.upload()
print("فایل‌های مرحله نهایی:", list(uploaded_final))


## ۶) ساخت خروجی نهایی

این سلول فارسی را با همان چیدمان دوزبانه روی ویدیو می‌چسباند، نسخهٔ کامل را می‌سازد و قسمت‌ها را دقیقاً با برش تأییدشده جدا می‌کند.


In [ ]:
import subprocess
from pathlib import Path

import yaml

# این خط‌ها اجازه می‌دهند بعد از ری‌استارت نشست هم فقط با اجرای سلول تنظیمات ادامه بدهید.
video_path = Path(VIDEO)
stem = video_path.stem
job_dir = Path("job")
job_dir.mkdir(exist_ok=True)
fa_srt_path = job_dir / f"{stem}.fa.srt"
cuts_path = job_dir / f"{stem}.cuts.suggested.txt"
AUTO_TRANSLATED = globals().get("AUTO_TRANSLATED", False)

fa_final = Path(FA_SRT_FINAL) if FA_SRT_FINAL else fa_srt_path
cuts_final = Path(APPROVED_CUTS) if APPROVED_CUTS else cuts_path

if not FA_SRT_FINAL and not AUTO_TRANSLATED:
    raise RuntimeError("فایل فارسی نهایی مشخص نشده است؛ FA_SRT_FINAL را نام فایل ترجمه‌شده بگذارید. قالب زبان مبدأ عمداً رندر نمی‌شود.")

for required in (video_path, fa_final, cuts_final):
    if not required.exists():
        raise FileNotFoundError(f"فایل لازم پیدا نشد: {required}")

render_config = {
    "output_mode": "original",
    "position": POSITION,
    "split_mode": "pause",
    "part_duration": PART_TARGET,
    "part_min_duration": PART_MIN,
    "part_max_duration": PART_MAX,
    "preset": PRESET,
    "crf": CRF,
    "font_dir": "fonts",
}
if MARGIN_V is not None:
    render_config["margin_v"] = MARGIN_V

render_config_path = job_dir / "render-config.yaml"
render_config_path.write_text(
    yaml.safe_dump(render_config, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)

cmd = [
    "fa-subs", "run", str(video_path),
    "--config", str(render_config_path),
    "--fa-srt", str(fa_final),
    "--cuts", str(cuts_final),
    "--output-dir", str(Path("output") / stem),
]
if not MAKE_FULL:
    cmd.append("--no-full")

print("▶", " ".join(cmd))
subprocess.run(cmd, check=True)


## ۷) پیش‌نمایش و دانلود خروجی‌ها

قسمت اول نمایش داده می‌شود و کل پوشهٔ خروجی این ویدیو به‌صورت ZIP دانلود می‌شود.


In [ ]:
import zipfile
from pathlib import Path
from IPython.display import Video, display

output_dir = Path("output") / stem
parts = sorted((output_dir / "parts").glob("*.mp4"))
print("قسمت‌ها:", [str(p) for p in parts])
if parts:
    display(Video(str(parts[0]), width=360))

outputs_zip = Path(f"{stem}-outputs.zip")
with zipfile.ZipFile(outputs_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(output_dir.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(output_dir.parent))
files.download(str(outputs_zip))
print("📦 خروجی‌ها دانلود شدند:", outputs_zip)
